In [ ]:
import pandas as pd
from IPython.display import display
from src.research_config import ResearchConfig
from src.fractional_OU import filter_antipersistent_pairs, compute_structural_t70, select_top_pairs_by_structural_t70


# 03 Pair Eligibility
Keep stable anti-persistent fOU fits, estimate structural convergence horizons and select up to 40 pairs.


In [ ]:
cfg = ResearchConfig()


In [ ]:
fou = pd.read_parquet("fractional_ou_parameters.parquet")
pool = filter_antipersistent_pairs(fou)
print(f"{len(pool)} of {len(fou)} fits pass model eligibility")
display(pool.head())


In [ ]:
structural = compute_structural_t70(
    pool,
    starting_z=cfg.entry_z,
    target_probability=cfg.target_probability,
    max_horizon_days=cfg.structural_horizon,
    n_paths=cfg.n_paths,
    seed=cfg.seed,
)
structural.to_parquet("structural_results.parquet")
display(structural[["pair", "structural_t70", "structural_probability_max"]].head(15))


In [ ]:
eligible_pool = structural.loc[structural.structural_t70.notna()].copy()
top_pairs = select_top_pairs_by_structural_t70(eligible_pool, cfg.top_n)
eligible_pool.to_parquet("eligible_pool.parquet")
top_pairs.to_parquet("eligible_pairs.parquet")
top_pairs.structural_t70.describe().to_frame("horizon").to_parquet("selected_horizon_summary.parquet")
print(f"{len(top_pairs)} selected pairs from {len(eligible_pool)} eligible pairs")
display(top_pairs[["pair", "hurst", "structural_t70", "structural_probability_max"]])
